In [ ]:
import pandas as pd

orders = pd.DataFrame({
    "order_id":[1,2,3,4,5,6,7,8,9,10],
    "customer_id":[101,101,101,102,102,103,103,103,103,104],
    "order_date":[
        "2024-01-01","2024-01-03","2024-01-10",
        "2024-01-02","2024-01-05",
        "2024-01-01","2024-01-03","2024-01-04","2024-01-10",
        "2024-01-07"
    ],
    "category":[
        "electronics","fashion","electronics",
        "grocery","fashion",
        "fashion","fashion","electronics","grocery",
        "electronics"
    ],
    "price":[100,50,200,20,80,40,60,150,30,500],
    "quantity":[1,2,1,3,1,2,1,1,5,1]
})

In [ ]:
orders.sort_values(by=['customer_id','order_date','order_id'], inplace=True)

orders['revenue'] = orders['price'] * orders['quantity']

orders['rank'] = orders.groupby('customer_id').cumcount() + 1

df_first3 = orders[orders['rank'] <= 3]

df_cat_wise = df_first3.groupby(['customer_id','category']).agg(
    cat_revenue=('revenue','sum')
).reset_index()

df_cat_wise['rank'] = df_cat_wise.groupby('customer_id')['cat_revenue'].rank(method='dense',ascending=False)
df_cat_wise = df_cat_wise[df_cat_wise['rank']==1]

In [ ]:
df_cat_wise

,customer_id,category,cat_revenue,rank
0,101,electronics,300,2.0
1,101,fashion,100,1.0
2,102,fashion,80,2.0
3,102,grocery,60,1.0
4,103,electronics,150,2.0
5,103,fashion,140,1.0
6,104,electronics,500,1.0


In [ ]:
df_cat_wise['rank'] = df_cat_wise.groupby('customer_id').cumcount() + 1
df_cat_wise = df_cat_wise[df_cat_wise['rank']==1]

,customer_id,category,cat_revenue
0,101,electronics,300
1,101,fashion,100
2,102,fashion,80
3,102,grocery,60
4,103,electronics,150
5,103,fashion,140
6,104,electronics,500


In [ ]:
events = pd.DataFrame({
"user_id":[1,1,1,1,2,2,2,3,3],
"event_date":[
"2024-01-01","2024-01-02","2024-01-05","2024-01-06",
"2024-01-01","2024-01-03","2024-01-10",
"2024-01-02","2024-01-03"
],
"spend":[10,20,5,40,15,25,30,8,12]
})

In [ ]:
events

,user_id,event_date,spend,prev_spend
0,1,2024-01-01,10,NaN
1,1,2024-01-02,20,10.0
2,1,2024-01-05,5,20.0
3,1,2024-01-06,40,5.0
4,2,2024-01-01,15,NaN
5,2,2024-01-03,25,15.0
6,2,2024-01-10,30,25.0
7,3,2024-01-02,8,NaN
8,3,2024-01-03,12,8.0


In [ ]:
events['event_date'] = pd.to_datetime(events['event_date'])
events.sort_values(by=['user_id','event_date'],inplace=True)
events['prev_spend'] = events.groupby('user_id')['spend'].shift(1)
events['days_since_last_spend'] = (events['event_date'] - events.groupby('user_id')['event_date'].shift(1)).dt.days
events['rolling_avg_3d_spend'] = events.groupby('user_id')['spend'].rolling(3).mean().reset_index(level=0,drop=True)

In [ ]:
events

,user_id,event_date,spend,prev_spend,days_since_last_spend,rolling_avg_3d_spend
0,1,2024-01-01,10,NaN,NaN,NaN
1,1,2024-01-02,20,10.0,1.0,NaN
2,1,2024-01-05,5,20.0,3.0,11.666667
3,1,2024-01-06,40,5.0,1.0,21.666667
4,2,2024-01-01,15,NaN,NaN,NaN
5,2,2024-01-03,25,15.0,2.0,NaN
6,2,2024-01-10,30,25.0,7.0,23.333333
7,3,2024-01-02,8,NaN,NaN,NaN
8,3,2024-01-03,12,8.0,1.0,NaN


In [ ]:
events.groupby('user_id')['spend'].rolling(3).mean().s

user_id   
1        0          NaN
         1          NaN
         2    11.666667
         3    21.666667
2        4          NaN
         5          NaN
         6    23.333333
3        7          NaN
         8          NaN
Name: spend, dtype: float64

In [ ]:
users = pd.DataFrame({
"user_id":[1,2,3,4,5],
"signup_date":[
"2024-01-01",
"2024-01-03",
"2024-01-10",
"2024-02-01",
"2024-02-05"
]
})
# users['signup_date'] = pd.to_datetime(users['signup_date'])
# users['signup_cohort'] = users['signup_date'].dt.strftime('%Y-%m')
users

,user_id,signup_date
0,1,2024-01-01
1,2,2024-01-03
2,3,2024-01-10
3,4,2024-02-01
4,5,2024-02-05


In [ ]:
purchases = pd.DataFrame({
"user_id":[1,1,2,2,2,3,4,4,5],
"purchase_date":[
"2024-01-01","2024-01-10",
"2024-01-05","2024-01-15","2024-02-01",
"2024-01-20",
"2024-02-03","2024-02-10",
"2024-02-07"
],
"revenue":[10,20,15,10,25,40,30,15,50]
})

purchases

,user_id,purchase_date,revenue
0,1,2024-01-01,10
1,1,2024-01-10,20
2,2,2024-01-05,15
3,2,2024-01-15,10
4,2,2024-02-01,25
5,3,2024-01-20,40
6,4,2024-02-03,30
7,4,2024-02-10,15
8,5,2024-02-07,50


In [ ]:
users['signup_date'] = pd.to_datetime(users['signup_date'])
users['signup_cohort'] = users['signup_date'].dt.to_period('M')
purchases['purchase_date'] = pd.to_datetime(purchases['purchase_date'])
purchases['purchase_cohort'] = purchases['purchase_date'].dt.to_period('M')
combined = pd.merge(users, purchases, on="user_id",how='left',suffixes=['_u','_p'])
combined['cohort_age'] = (combined['purchase_month'] - combined['signup_month'])
combined.pivot_table(
    index='signup_month',
    columns='cohort_age',
    values = 'user_id',
    aggfunc = 'nunique'
)

cohort_age,0,1
signup_month,,
1,3.0,1.0
2,2.0,NaN


In [ ]:
combined.pivot_table(
    index='signup_cohort',
    columns='purchase_cohort',
    values = 'user_id',
    aggfunc = 'nunique'
)

purchase_cohort,2024-01,2024-02
signup_cohort,,
2024-01,3.0,1.0
2024-02,NaN,2.0


In [ ]:
sales = pd.DataFrame({
"store":[
"A","A","A","A",
"B","B","B",
"C","C","C"
],
"date":[
"2024-01-01","2024-01-02","2024-01-03","2024-01-04",
"2024-01-01","2024-01-02","2024-01-03",
"2024-01-01","2024-01-02","2024-01-03"
],
"revenue":[
100,None,120,None,
200,None,210,
None,50,None
]
})

In [ ]:
sales

,store,date,revenue
0,A,2024-01-01,100.0
1,A,2024-01-02,NaN
2,A,2024-01-03,120.0
3,A,2024-01-04,NaN
4,B,2024-01-01,200.0
5,B,2024-01-02,NaN
6,B,2024-01-03,210.0
7,C,2024-01-01,NaN
8,C,2024-01-02,50.0
9,C,2024-01-03,NaN


In [ ]:
store_mean = sales.groupby('store').agg(store_mean=('revenue','mean')).reset_index()
store_mean

,store,store_mean
0,A,110.0
1,B,205.0
2,C,50.0


In [ ]:
store_rolling_3d = sales.groupby('store')['revenue'].rolling(3).mean()
store_rolling_3d

store   
A      0   NaN
       1   NaN
       2   NaN
       3   NaN
B      4   NaN
       5   NaN
       6   NaN
C      7   NaN
       8   NaN
       9   NaN
Name: revenue, dtype: float64

In [ ]:
df = pd.DataFrame({'Math': [90, 88], 'Science': [88, 92]})
melted = pd.melt(df)
# Output: variable, value columns


In [ ]:
melted

,variable,value
0,Math,90
1,Math,88
2,Science,88
3,Science,92


In [ ]:
df = pd.DataFrame({'Name': ['A', 'B'], 'Math': [90, 88], 'Sci': [88, 92]})
melted = pd.melt(df, id_vars=['Name'])
# Output: Name, variable, value
melted

,Name,variable,value
0,A,Math,90
1,B,Math,88
2,A,Sci,88
3,B,Sci,92


In [ ]:
df

,Name,Math,Sci
0,A,90,88
1,B,88,92
